In [2]:
import os 
import sys
import pandas as pd 
import numpy as np 

OR_LEARNING_PATH = os.path.join(os.getcwd().split('OR_learning')[0], 'OR_learning/')
sys.path.insert(0, os.path.join(OR_LEARNING_PATH, 'utils/'))

import BindingCavity_functions as bc 
import SequenceAlignment_functions as sa
import plot_functions as pf
import color_function as cf 
import pdb_functions as pu

In [ ]:
# TODO include mutations from structure paper. See how that influences the binding cavity shape. 
# TODO also conduct similar pipeline for validating res of Or51e2 but using consOR52 as HM template. 

### Modeller - Homology Model 

Use pykvFinder and test whether defined cavity and identified residues includes the experimentally confirmed resiudes for Or51E2<br>

OR51E2 residues within 5A from propionate<br>
`H104, F155, L158, H180, Q181, N194, G198, L199, A201, I202, S258, R262`<br>
<br>
`R262` is ClassI-conserved residue crucial for carboxylic acid binding<br>
`F155` is known to be important for determining the size of carboxylic acid ligands<br>

Test how different Homology model may be improvement for using structural analysis to identify binding cavity and cavity residues <br>
Things to test consists of: 
- Testing HM_Or51E2 against different mutations of HM_Or51E2 in the structural paper. This can potentially recapitulate the idea that binding cavity shape and interacting residues changes by mutation. And more importantly can be shown in via homology model 
- Conduct similar pipeline of comparing HM_Or51E2 against cryo_OR51E2, but the HM should be built on consOR51 as a template. 

#### HM_Or51E2 with cons51 as template 

In [ ]:
"""
Conduct Structural Sequence Alignment to generate alignment .fa for modeller
protein to get aligned primary sequence for modeller 
"""

ref_pdb = pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51.pdb')
tgt_pdb = pu.load_pdb_coordinates('/mnt/data2/Justice/AF_files/AF_tmaligned_pdb/Or51E2_Mol2.3_Olfr78_Psgr_tmaligned.pdb')

# tmlign backbone in order to perform structural alignment
_, ref_backbone = pu.tmalign_pdb("/mnt/data2/Justice/AF_files/AF_pdb/Olfr1377.pdb", 
                                 "/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51.pdb",             
                                 save_pdb = False, return_coords=True)
ref_sequence = ref_pdb[2]
tgt_backbone, tgt_sequence = tgt_pdb[1], tgt_pdb[2]

alignment = sa.structural_alignment_dp(ref_backbone, ref_sequence, 
                                       tgt_backbone, tgt_sequence, 
                                       gap_penalty=10)
for _seq in alignment: 
    print(_seq)
    
# Generate .fa file for modeller 
ref_aligned_seq, tgt_aligned_seq = alignment[0], alignment[1]
pu.write_modeller_fa(tgt_aligned_seq,
                     ref_aligned_seq,
                     ref_pdb = '/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51.pdb',
                     labels = ['Or51E2', 'consOR51'], 
                     output_file = '/mnt/data2/Justice/OR_learning/files/modeller/fa_alignment/Or51E2_consOR51.fa')

In [ ]:
"""
Run homology model modeller 
"""

pu.run_modeller_homology(aln_file = '/mnt/data2/Justice/OR_learning/files/modeller/fa_alignment/Or51E2_consOR51.fa', 
                         template_pdb = 'consOR51',
                         target_name = 'Or51E2',
                         atom_files_directory=[ '/mnt/data2/Justice/OR_learning/files/OR_seq/'],
                         output_dir=os.path.join('/mnt/data2/Justice/OR_learning/files/modeller/Or51E2_consOR51'), 
                         num_models=3)

In [5]:
"""
Re align generated homology models to reference (Olfr1377). 

So that it is consistent with all the other pdb in AF_tmalign. 
"""


modeller_OUTPUT = '/mnt/data2/Justice/OR_learning/files/modeller/Or51E2_consOR51'

# Add HM files
pdb_files = [os.path.join(modeller_OUTPUT, _files) for _files in os.listdir(modeller_OUTPUT) if (_files.endswith('.pdb') & 
                                                                                     ('tmaligned' not in _files))]
# Add AF3 file
pdb_files.append('/mnt/data2/Justice/OR_learning/files/TEST_modeller/Or51E2_AF3/AF3_Or51E2.pdb')

# Re align homology model pdb to reference 
tmaligned_pdb = {}
for _pdb in pdb_files: 
    name = _pdb.split('/')[-1].replace('.pdb','')
    tmaligned_pdb[name] = pu.tmalign_pdb("/mnt/data2/Justice/AF_files/AF_pdb/Olfr1377.pdb", 
                                        _pdb, _pdb.replace('.pdb', '_tmaligned.pdb'), 
                                        return_coords=True
                                        )
                         

In [ ]:
tmaligned_pdb['AF3_Or51E2'] = bc.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/TEST_modeller/Or51E2_AF3/AF3_Or51E2_tmaligned.pdb')
tmaligned_pdb['cryo_AF_Or51E2'] = bc.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb')

# Plot
colormap = cf.distinct_colors(list(tmaligned_pdb.keys()))
fig = pf.plot_coordinates([tmaligned_pdb[_OR][1] for _OR in tmaligned_pdb], 
                          labels=list(tmaligned_pdb.keys()), 
                          colors=colormap, opacity=0.5, 
                          marker_size=8,
                          mode='markers')

fig.show()

In [6]:
"""
Generate pyKV cavity, cavity surface and residue coordinates 
"""

modeller_OUTPUT = '/mnt/data2/Justice/OR_learning/files/modeller/Or51E2_consOR51'
pdb_files = [os.path.join(modeller_OUTPUT, _files) for _files in os.listdir(modeller_OUTPUT) if (_files.endswith('.pdb') & 
                                                                                                 ('tmaligned' in _files))]
pdb_files += ['/mnt/data2/Justice/OR_learning/files/TEST_modeller/AF2_Or51E2_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/Or51E2_AF3/AF3_Or51E2_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb']

cav_coords, cavsurf_coords, res_coords = bc.run_pyKVFinder_workflow(pdb_files)

OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested

In [7]:
"""
Filter the cavity and cavity res via defined canonical binding pocket zone to see what is preserved 
"""

OR_LEARNING_PATH = os.path.join(os.getcwd().split('OR_learning')[0], 'OR_learning/')
canonical_bc_coords = pd.read_pickle(os.path.join(OR_LEARNING_PATH,'files/binding_cavity/canonical_bc_coords.pkl'))

# Filtering ORs cavity and residue coordinates if they overlap with the defined canonical binding cavity
Cbc_cav_coords = { _Or: bc.filter_coordinates_within_cavity(canonical_bc_coords, 
                                                             np.array(cavsurf_coords[_Or])) for _Or in cavsurf_coords}

Cbc_res_coords = { _Or: bc.filter_coordinates_within_cavity(canonical_bc_coords, 
                                                             np.array(res_coords[_Or]), 
                                                             is_residue=True) for _Or in res_coords}


In [8]:
"""
Quick print statements to check the identified cavity binding residue overlaps with OBR 
"""

modeller_OUTPUT = '/mnt/data2/Justice/OR_learning/files/modeller/Or51E2_consOR51'
pdb_files = [os.path.join(modeller_OUTPUT, _files) for _files in os.listdir(modeller_OUTPUT) if (_files.endswith('.pdb') & 
                                                                                                 ('tmaligned' in _files))]
pdb_files += ['/mnt/data2/Justice/OR_learning/files/TEST_modeller/AF2_Or51E2_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/Or51E2_AF3/AF3_Or51E2_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb']

ref_pdb = '/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb'

alignment_pairs = sa.generate_sequence_alignment_pairs_fromPDB(ref_pdb,
                                                               pdb_files, 
                                                               pu.load_pdb_coordinates, 
                                                               labels = list(Cbc_res_coords.keys()), 
                                                               gap_penalty=100)
alignment = sa.union_gaps_with_consistency(alignment_pairs)




Or51E2_bw_positions = {"N41":'1.50', "D69": '2.50', "R121":'3.50', 
                       "V149":'4.50', "C178":'45.50', "D209":'5.50', 
                       "P253":'6.50', "P288":'7.50'}

OBR = ['H104', 'F155', 'L158', 'H180', 'Q181', 'N194', 'G198', 'L199', 'A201', 'I202', 'S258', 'R262']

alignment_bw = sa.update_bw_positions("AF2_Or51E2", alignment, Or51E2_bw_positions)

# Translate the OBR to bw number for comparison 
OBR_bw = sa.map_residues_to_bw(OBR,
                               alignment, "AF2_Or51E2", alignment_bw)

for _OR in Cbc_res_coords:     
    mapped_bw = sa.map_residues_to_bw(np.unique(Cbc_res_coords[_OR][:,0]),
                                alignment, _OR, 
                                alignment_bw)
    
    if _OR == 'consOR51': # Exceptions these pdb starts at x residue position, NOT THE FULL LENGTH
        mapped_bw = sa.map_residues_to_bw([str(int(i)-10) for i in np.unique(Cbc_res_coords[_OR][:,0])],
                                alignment, _OR, 
                                alignment_bw)
    if _OR == 'cryo_OR51E2': 
        mapped_bw = sa.map_residues_to_bw([str(int(i)-3) for i in np.unique(Cbc_res_coords[_OR][:,0])],
                                alignment, _OR, 
                                alignment_bw)    
    
    print(f'\n\n{_OR} . . . ')
    sa.compare_bw_numbers(mapped_bw, OBR_bw)


    



HM_Or51E2_1 . . . 
Target BW    Overlapping BW    Missing Query BW
-----------  ----------------  ------------------
I68: 2.49    H104: 3.33        N194: 5.35
D69: 2.5     F155: 4.56
S73: 2.54    L158: 4.59
T76: 2.57    H180: 45.52
I103: 3.32   Q181: 45.53
H104: 3.33   G198: 5.39
T105: 3.34   L199: 5.4
S107: 3.36   A201: 5.42
A108: 3.37   I202: 5.43
I109: 3.38   S258: 6.55
S111: 3.4    R262: 6.59
R150: 4.51
G151: 4.52
F154: 4.55
F155: 4.56
L158: 4.59
S174: 45.46
S176: 45.48
Y177: 45.49
V179: 45.51
H180: 45.52
Q181: 45.53
G198: 5.39
L199: 5.4
A201: 5.42
I202: 5.43
V205: 5.46
M206: 5.47
D209: 5.5
F250: 6.47
L254: 6.51
L257: 6.54
S258: 6.55
R262: 6.59
V273: 7.35
D277: 7.39
Y279: 7.41
L280: 7.42
L281: 7.43
P284: 7.46


HM_Or51E2_2 . . . 
Target BW    Overlapping BW    Missing Query BW
-----------  ----------------  ------------------
D69: 2.5     H104: 3.33        L199: 5.4
L72: 2.53    F155: 4.56
T76: 2.57    L158: 4.59
M100: 3.29   H180: 45.52
I103: 3.32   Q181: 45.53
H104: 3.33   N194

In [ ]:
import re 

AF2_Or51E2 = bc.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/TEST_modeller/AF2_Or51E2_tmaligned.pdb')[1]
# cryo_OR51E2 = bc.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb')[1]

cryo_OR51E2 = pu.tmalign_pdb('/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb',
                              '/mnt/data2/Justice/OR_learning/files/TEST_modeller/AF2_Or51E2_tmaligned.pdb',
                              save_pdb=False, 
                              return_coords = True)[1]

plot_data = []

plot_data += [AF2_Or51E2, cryo_OR51E2] # Adding backbone coordinates 

# Add cavity contact residues 
plot_data += [[AF2_Or51E2[int(re.sub(r"\D", "", _res))] for _res in OBR_bw]]
plot_data += [[cryo_OR51E2[int(re.sub(r"\D", "", _res))] for _res in mapped_bw]]

# Plot
colormap = cf.distinct_colors([1,2], form='list')
colormap += colormap

fig = pf.plot_coordinates(plot_data, 
                          labels=['AF2_Or51E2', 'cryo_OR51E2', 'AF2_Or51E2_res', 'cryo_OR51E2_res'], 
                          colors=colormap, 
                          opacity=[0.3, 0.3, 0.8, 0.8], 
                          size=[5, 5, 10, 10],
                          mode='markers')
fig.show()

In [ ]:
import voxel_functions as vf 

# Add cavity coordinates
plot_data = Cbc_cav_coords.copy()

# Add residue coordinates 
for _OR in Cbc_res_coords: 
    arr = np.array(Cbc_res_coords[_OR])
    arr = arr[arr[:, 2] == 'CA']
    plot_data[f'{_OR}_res'] = arr[np.argsort(arr[:, 0].astype(int))][:,3:6]
    
plot_data['background'] = pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb')[1]
offset = 4 # From converting res based on full length OR51E2 to cryo_OR51E2 (starts at res 4).
OBR = ['H104', 'F155', 'L158', 'H180', 'Q181', 'N194', 'G198', 'L199', 'A201', 'I202', 'S258', 'R262']
plot_data['OBR'] = [list(plot_data['background'][int(_res[1:4])-offset]) for _res in OBR]

# Standardize dtype in dict
for _key in plot_data:
    plot_data[_key] = np.array(plot_data[_key], dtype=np.float64)


voxelized_cavities, voxel_shape = vf.voxelize_cavity(list(plot_data.values()), 
                                                    #  list(Cbc_res_coords.values()), 
                                                     resolution=1)

# Output: List of 1D arrays representing voxelized space
print(np.array(voxelized_cavities).shape)

labels = [f"{i}" for i in plot_data.keys()]
color_map = cf.distinct_colors([_key for _key in plot_data.keys() if '_res' not in _key])
# Assign same color for Or ans res 

for _key in list(color_map.keys()): 
    if 'or' in _key.lower(): 
        color_map[f'{_key}_res'] = color_map[_key]

color_map['background'] = '#D3D3D3'

fig = pf.visualize_voxel_grid(voxelized_cavities, 
                        labels, 
                        color_map, 
                        size=[10 if _label == 'OBR' else 5 for _label in labels],
                        highlight_labels = [_label for _label in labels if '_res' in _label ],
                        highlight_opacity=0.7,
                        opacity=0.2)

fig.update_layout(title='Homology Model moideller Or51E2 with consOR51 template', 
                          margin=dict(r=10, l=10, b=10, t=30))
fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/modeller/HM_Or51E2_consOR51/_Or51E2_CBC_comparisons.html')

### Or51E2 mutants HM

In [ ]:
"""
OR51E1 Mutants changed 
    M158A and I205A

"""

In [21]:
"""
Generate alignment 

Create mutant sequence by directly mutating the aligned sequence and generate .fa file for modeller 
"""

ref_pdb = pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51.pdb')
tgt_pdb = pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/modeller/AF2_OR51E1.pdb')

# tmlign backbone in order to perform structural alignment
_, ref_backbone = pu.tmalign_pdb('/mnt/data2/Justice/OR_learning/files/modeller/AF2_OR51E1.pdb', 
                                 '/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51.pdb',
                                 save_pdb = False, 
                                 return_coords=True)

ref_sequence = ref_pdb[2]
tgt_backbone, tgt_sequence = tgt_pdb[1], tgt_pdb[2]

alignment = sa.structural_alignment_dp(ref_backbone, ref_sequence, 
                                       tgt_backbone, tgt_sequence, 
                                       gap_penalty=10)
ref_aligned_seq, tgt_aligned_seq = alignment[0], alignment[1]

# Generate .fa file for each mutant 
mutant_res = ['', 'M158A', 'I205A']
for _mut in mutant_res: 
    seq = tgt_aligned_seq
    if len(_mut) > 1: 
        orig_aa, mut_aa, mut_pos = _mut[0], _mut[-1], int(_mut[1:len(_mut)-1])-1 
        mut_seq = seq[:mut_pos] + mut_aa + seq[mut_pos + 1:]
    else: 
        mut_seq = seq 
        
    pu.write_modeller_fa(mut_seq,
                            ref_aligned_seq,
                            ref_pdb = '/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51.pdb',
                            labels = [f'OR51E1_{_mut}', 'consOR51'], 
                            output_file = f'/mnt/data2/Justice/OR_learning/files/modeller/fa_alignment/OR51E1_{_mut}_consOR51.fa')


Alignment file written to /mnt/data2/Justice/OR_learning/files/modeller/fa_alignment/OR51E1__consOR51.fa
Alignment file written to /mnt/data2/Justice/OR_learning/files/modeller/fa_alignment/OR51E1_M158A_consOR51.fa
Alignment file written to /mnt/data2/Justice/OR_learning/files/modeller/fa_alignment/OR51E1_I205A_consOR51.fa


In [22]:
"""
Run homology model modeller 
"""

modeller_files = ['OR51E1_', 'OR51E1_M158A', 'OR51E1_I205A']
for _file in modeller_files: 
    pu.run_modeller_homology(aln_file = f'/mnt/data2/Justice/OR_learning/files/modeller/fa_alignment/{_file}_consOR51.fa', 
                            template_pdb = 'consOR51.pdb',
                            target_name = _file,
                            atom_files_directory=[ '/mnt/data2/Justice/OR_learning/files/OR_seq/'],
                            output_dir=os.path.join(f'/mnt/data2/Justice/OR_learning/files/modeller/{_file}_consOR51'), 
                            num_models=1)


                         MODELLER 10.6, 2024/10/17, r12888

     PROTEIN STRUCTURE MODELLING BY SATISFACTION OF SPATIAL RESTRAINTS


                     Copyright(c) 1989-2024 Andrej Sali
                            All Rights Reserved

                             Written by A. Sali
                               with help from
              B. Webb, M.S. Madhusudhan, M-Y. Shen, G.Q. Dong,
          M.A. Marti-Renom, N. Eswar, F. Alber, M. Topf, B. Oliva,
             A. Fiser, R. Sanchez, B. Yerkovich, A. Badretdinov,
                     F. Melo, J.P. Overington, E. Feyfant
                 University of California, San Francisco, USA
                    Rockefeller University, New York, USA
                      Harvard University, Cambridge, USA
                   Imperial Cancer Research Fund, London, UK
              Birkbeck College, University of London, London, UK


Kind, OS, HostName, Kernel, Processor: 4, Linux MGM-GPU 6.8.0-49-generic x86_64
Date and time of compilation 

In [24]:
"""
Re align generated homology models to reference (Olfr1377). 

So that it is consistent with all the other pdb in AF_tmalign. 
"""

# Add HM files
pdb_files = ['/mnt/data2/Justice/OR_learning/files/modeller/AF2_OR51E1.pdb', 
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1__consOR51/HM_OR51E1_.B99990001.pdb', 
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1_I205A_consOR51/HM_OR51E1_I205A.B99990001.pdb', 
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1_M158A_consOR51/HM_OR51E1_M158A.B99990001.pdb', 
             '/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51.pdb']

# Re align homology model pdb to reference 
tmaligned_pdb = {}
for _pdb in pdb_files: 
    name = _pdb.split('/')[-1].replace('.pdb','')
    tmaligned_pdb[name] = pu.tmalign_pdb("/mnt/data2/Justice/AF_files/AF_pdb/Olfr1377.pdb", 
                                        _pdb, _pdb.replace('.pdb', '_tmaligned.pdb'), 
                                        save_pdb=True, return_coords=True
                                        )
                         

Aligned PDB saved to: /mnt/data2/Justice/OR_learning/files/modeller/AF2_OR51E1_tmaligned.pdb
Aligned PDB saved to: /mnt/data2/Justice/OR_learning/files/modeller/OR51E1__consOR51/HM_OR51E1_.B99990001_tmaligned.pdb
Aligned PDB saved to: /mnt/data2/Justice/OR_learning/files/modeller/OR51E1_I205A_consOR51/HM_OR51E1_I205A.B99990001_tmaligned.pdb
Aligned PDB saved to: /mnt/data2/Justice/OR_learning/files/modeller/OR51E1_M158A_consOR51/HM_OR51E1_M158A.B99990001_tmaligned.pdb
Aligned PDB saved to: /mnt/data2/Justice/OR_learning/files/OR_seq/consOR51_tmaligned.pdb


In [ ]:
tmaligned_pdb['cryo_AF_Or51E2'] = bc.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb')

# Plot
colormap = cf.distinct_colors(list(tmaligned_pdb.keys()))
fig = pf.plot_coordinates([tmaligned_pdb[_OR][1] for _OR in tmaligned_pdb], 
                          labels=list(tmaligned_pdb.keys()), 
                          colors=colormap, opacity=0.5, 
                          size=8,
                          mode='markers')

fig.show()
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/modeller/OR51E1_mutant/OR51E1_mutants.html')

In [28]:
"""
Generate pyKV cavity, cavity surface and residue coordinates 
"""

pdb_files = ['/mnt/data2/Justice/OR_learning/files/modeller/AF2_OR51E1_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/AF2_Or51E2_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1__consOR51/HM_OR51E1_.B99990001_tmaligned.pdb', 
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1_I205A_consOR51/HM_OR51E1_I205A.B99990001_tmaligned.pdb', 
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1_M158A_consOR51/HM_OR51E1_M158A.B99990001_tmaligned.pdb', 
             '/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb']

cav_coords, cavsurf_coords, res_coords = bc.run_pyKVFinder_workflow(pdb_files)

OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested

In [29]:
"""
Filter the cavity and cavity res via defined canonical binding pocket zone to see what is preserved 
"""

OR_LEARNING_PATH = os.path.join(os.getcwd().split('OR_learning')[0], 'OR_learning/')
canonical_bc_coords = pd.read_pickle(os.path.join(OR_LEARNING_PATH,'files/binding_cavity/canonical_bc_coords.pkl'))

# Filtering ORs cavity and residue coordinates if they overlap with the defined canonical binding cavity
Cbc_cav_coords = { _Or: bc.filter_coordinates_within_cavity(canonical_bc_coords, 
                                                             np.array(cavsurf_coords[_Or])) for _Or in cavsurf_coords}

Cbc_res_coords = { _Or: bc.filter_coordinates_within_cavity(canonical_bc_coords, 
                                                             np.array(res_coords[_Or]), 
                                                             is_residue=True) for _Or in res_coords}


In [ ]:
import voxel_functions as vf 

# Add cavity coordinates
plot_data = Cbc_cav_coords.copy()

# Add residue coordinates 
for _OR in Cbc_res_coords: 
    arr = np.array(Cbc_res_coords[_OR])
    arr = arr[arr[:, 2] == 'CA']
    plot_data[f'{_OR}_res'] = arr[np.argsort(arr[:, 0].astype(int))][:,3:6]
    
plot_data['background'] = pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb')[1]
offset = 4 # From converting res based on full length OR51E2 to cryo_OR51E2 (starts at res 4).
OBR = ['H104', 'F155', 'L158', 'H180', 'Q181', 'N194', 'G198', 'L199', 'A201', 'I202', 'S258', 'R262']
plot_data['OBR'] = [list(plot_data['background'][int(_res[1:4])-offset]) for _res in OBR]

OR51E1_RES = ['H107', 'G111', 'M158', 'L161', 'H183', 'Q184', 'V204', 'I205', 'S260', 'R264']
temp_OR51E1 = pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/modeller/AF2_OR51E1_tmaligned.pdb')[1]
plot_data['OR51E1_RES'] = [list(temp_OR51E1[int(_res[1:4]) - 1]) for _res in OR51E1_RES]

# Standardize dtype in dict
for _key in plot_data:
    plot_data[_key] = np.array(plot_data[_key], dtype=np.float64)


voxelized_cavities, voxel_shape = vf.voxelize_cavity(list(plot_data.values()), 
                                                    #  list(Cbc_res_coords.values()), 
                                                     resolution=1)

# Output: List of 1D arrays representing voxelized space
print(np.array(voxelized_cavities).shape)

labels = [f"{i}" for i in plot_data.keys()]
color_map = cf.distinct_colors([_key for _key in plot_data.keys() if '_res' not in _key])
# Assign same color for Or ans res 

for _key in list(color_map.keys()): 
    if 'or' in _key.lower(): 
        color_map[f'{_key}_res'] = color_map[_key]

color_map['background'] = '#D3D3D3'

fig = pf.visualize_voxel_grid(voxelized_cavities, 
                        labels, 
                        color_map, 
                        size=[10 if _label in ['OBR', 'OR51E1_RES'] else 5 for _label in labels],
                        highlight_labels = [_label for _label in labels if '_res' in _label ],
                        highlight_opacity=0.7,
                        opacity=0.2)

fig.update_layout(title='Homology Model moideller Or51E2 mutants with consOR51 template', 
                          margin=dict(r=10, l=10, b=10, t=30))
fig.show()
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/modeller/OR51E1_mutant/OR51E1_mutant_CBC_comparisons.html')

In [270]:
"""
Quick print statements to check the identified cavity binding residue overlaps with OBR 
"""

pdb_files = ['/mnt/data2/Justice/OR_learning/files/modeller/AF2_OR51E1.pdb', 
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/AF2_Or51E2_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1_I205A_consOR51/HM_OR51E1_I205A.B99990001_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1_M158A_consOR51/HM_OR51E1_M158A.B99990001_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb']

ref_pdb = '/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb'

# labels = [_files.split('/')[-1].split('.')[0].replace('_tmaligned', '') for _files in pdb_files]

# Align sequences to generate MSA for extracting aligned residue positions 
alignment_pairs = sa.generate_sequence_alignment_pairs_fromPDB(ref_pdb,
                                                               pdb_files, 
                                                               pu.load_pdb_coordinates, 
                                                               labels = list(Cbc_res_coords.keys()), 
                                                               gap_penalty=100)
alignment = sa.union_gaps_with_consistency(alignment_pairs)

Or51E2_bw_positions = {"N41":'1.50', "D69": '2.50', "R121":'3.50', 
                       "V149":'4.50', "C178":'45.50', "D209":'5.50', 
                       "P253":'6.50', "P288":'7.50'}
OBR = ['H104', 'F155', 'L158', 'H180', 'Q181', 'N194', 'G198', 'L199', 'A201', 'I202', 'S258', 'R262']

alignment_bw = sa.update_bw_positions("AF2_Or51E2", alignment, Or51E2_bw_positions)

# Translate the OBR to bw number for comparison 
OBR_bw = sa.map_residues_to_bw(OBR,
                               alignment, "AF2_Or51E2", alignment_bw)

for _OR in Cbc_res_coords:     
    mapped_bw = sa.map_residues_to_bw(np.unique(Cbc_res_coords[_OR][:,0]),
                                alignment, _OR, 
                                alignment_bw)
    
    if _OR == 'consOR51': # Exceptions these pdb starts at x residue position, NOT THE FULL LENGTH
        mapped_bw = sa.map_residues_to_bw([str(int(i)-10) for i in np.unique(Cbc_res_coords[_OR][:,0])],
                                alignment, _OR, 
                                alignment_bw)
    if _OR == 'cryo_OR51E2': 
        mapped_bw = sa.map_residues_to_bw([str(int(i)-3) for i in np.unique(Cbc_res_coords[_OR][:,0])],
                                alignment, _OR, 
                                alignment_bw)    
    
    print(f'\n\n{_OR} . . . ')
    sa.compare_bw_numbers(mapped_bw, OBR_bw)


    



OR51E1 . . . 
Target BW    Overlapping BW    Missing Query BW
-----------  ----------------  ------------------
I71: 2.49    H107: 3.33        S258: 6.55
I75: 2.53    M158: 4.56        R262: 6.59
M103: 3.29   L161: 4.59
F104: 3.3    H183: 45.52
I106: 3.32   Q184: 45.53
H107: 3.33   N197: 5.35
S110: 3.36   G201: 5.39
G111: 3.37   L202: 5.4
M112: 3.38   V204: 5.42
E113: 3.39   I205: 5.43
R153: 4.51
G154: 4.52
L157: 4.55
M158: 4.56
L161: 4.59
P162: 4.6
S177: 45.46
H178: 45.47
Y180: 45.49
L182: 45.51
H183: 45.52
Q184: 45.53
D185: 45.54
M187: 45.56
K188: 45.57
L189: 45.58
N197: 5.35
V198: 5.36
G201: 5.39
L202: 5.4
V204: 5.42
I205: 5.43
A208: 5.46
I209: 5.47
F252: 6.46
F256: 6.5
L259: 6.53
S260: 6.54
V262: 6.56
H263: 6.57
P274: 6.68
V275: 6.69
N279: 7.35
Y281: 7.37
L282: 7.38
P285: 7.41
P286: 7.42


AF2_Or51E2 . . . 
Target BW    Overlapping BW    Missing Query BW
-----------  ----------------  ------------------
I68: 2.49    H104: 3.33        F155: 4.56
D69: 2.5     L158: 4.59        A201

In [35]:
"""
Quick print statements to check the identified cavity binding residue overlaps with defined OR51E1 residue  
`https://www.nature.com/articles/s41586-024-08126-0/figures/2`
"""

pdb_files = ['/mnt/data2/Justice/OR_learning/files/modeller/AF2_OR51E1.pdb', 
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/AF2_Or51E2_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1__consOR51/HM_OR51E1_.B99990001_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1_I205A_consOR51/HM_OR51E1_I205A.B99990001_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/modeller/OR51E1_M158A_consOR51/HM_OR51E1_M158A.B99990001_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/OR_seq/consOR51_tmaligned.pdb',
             '/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb']

ref_pdb = '/mnt/data2/Justice/OR_learning/files/TEST_modeller/cryo_OR51E2_tmaligned.pdb'

# labels = [_files.split('/')[-1].split('.')[0].replace('_tmaligned', '') for _files in pdb_files]

# Align sequences to generate MSA for extracting aligned residue positions 
alignment_pairs = sa.generate_sequence_alignment_pairs_fromPDB(ref_pdb,
                                                               pdb_files, 
                                                               pu.load_pdb_coordinates, 
                                                               labels = list(Cbc_res_coords.keys()), 
                                                               gap_penalty=100)
alignment = sa.union_gaps_with_consistency(alignment_pairs)

Or51E2_bw_positions = {"N41":'1.50', "D69": '2.50', "R121":'3.50', 
                       "V149":'4.50', "C178":'45.50', "D209":'5.50', 
                       "P253":'6.50', "P288":'7.50'}

OR51E1_RES = ['H107', 'G111', 'L161', 'M158', 'H183', 'Q184', 'V204', 'I205', 'S260', 'R264']


alignment_bw = sa.update_bw_positions("AF2_Or51E2", alignment, Or51E2_bw_positions)

# Translate the OBR to bw number for comparison 
OBR_bw = sa.map_residues_to_bw(OR51E1_RES,
                               alignment, "AF2_OR51E1", alignment_bw)

for _OR in Cbc_res_coords:     
    mapped_bw = sa.map_residues_to_bw(np.unique(Cbc_res_coords[_OR][:,0]),
                                alignment, _OR, 
                                alignment_bw)
    
    if _OR == 'consOR51': # Exceptions these pdb starts at x residue position, NOT THE FULL LENGTH
        mapped_bw = sa.map_residues_to_bw([str(int(i)-10) for i in np.unique(Cbc_res_coords[_OR][:,0])],
                                alignment, _OR, 
                                alignment_bw)
    if _OR == 'cryo_OR51E2': 
        mapped_bw = sa.map_residues_to_bw([str(int(i)-3) for i in np.unique(Cbc_res_coords[_OR][:,0])],
                                alignment, _OR, 
                                alignment_bw)    
    
    print(f'\n\n{_OR} . . . ')
    sa.compare_bw_numbers(mapped_bw, OBR_bw)


    



AF2_OR51E1 . . . 
Target BW    Overlapping BW    Missing Query BW
-----------  ----------------  ------------------
I71: 2.49    H107: 3.33        R264: 6.58
I75: 2.53    G111: 3.37
M103: 3.29   M158: 4.56
F104: 3.3    L161: 4.59
I106: 3.32   H183: 45.52
H107: 3.33   Q184: 45.53
S110: 3.36   V204: 5.42
G111: 3.37   I205: 5.43
M112: 3.38   S260: 6.54
E113: 3.39
R153: 4.51
G154: 4.52
L157: 4.55
M158: 4.56
L161: 4.59
P162: 4.6
S177: 45.46
H178: 45.47
Y180: 45.49
L182: 45.51
H183: 45.52
Q184: 45.53
D185: 45.54
M187: 45.56
K188: 45.57
L189: 45.58
N197: 5.35
V198: 5.36
G201: 5.39
L202: 5.4
V204: 5.42
I205: 5.43
A208: 5.46
I209: 5.47
F252: 6.46
F256: 6.5
L259: 6.53
S260: 6.54
V262: 6.56
H263: 6.57
P274: 6.68
V275: 6.69
N279: 7.35
Y281: 7.37
L282: 7.38
P285: 7.41
P286: 7.42


AF2_Or51E2 . . . 
Target BW    Overlapping BW    Missing Query BW
-----------  ----------------  ------------------
I68: 2.49    H104: 3.33        G111: 3.37
D69: 2.5     L158: 4.59        M158: 4.56
L72: 2.53    H180: 

In [269]:
import importlib 
importlib.reload(pu)
importlib.reload(pf)
importlib.reload(bc)
importlib.reload(sa)

<module 'SequenceAlignment_functions' from '/mnt/data2/Justice/OR_learning/utils/SequenceAlignment_functions.py'>